# 8. Final statistical analysis

Shared 5-fold, leader-approved post-result protocol: XGBoost is compared with Random Forest, Logistic Regression, Decision Tree, KNN, and Naive Bayes. The paired difference is `XGBoost log loss - comparator log loss`; Bonferroni family size is 5.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Run from the repository root or notebooks directory.')
sys.path.insert(0, str(PROJECT_ROOT))

from src import p4_analysis, stats
print('Project root:', PROJECT_ROOT)

## 1. Load and validate final scores

In [ ]:
scores = pd.read_csv(PROJECT_ROOT / 'results' / 'scores_all.csv')
score_validation = p4_analysis.validate_scores(scores)
score_validation

In [ ]:
performance_summary = scores.groupby('model')[['log_loss', 'accuracy', 'macro_f1']].agg(['mean', 'std'])
performance_summary.sort_values(('log_loss', 'mean'))

## 2. HEP-13 — paired tests, confidence intervals, Bonferroni correction

In [ ]:
hep13 = p4_analysis.run_hep13(scores)
hep13

A confidence interval entirely below zero supports lower fold-level log loss for XGBoost. A confidence interval containing zero would indicate insufficient evidence, not equivalence.

## 3. HEP-14 — Shapiro-Wilk and Wilcoxon sensitivity analysis

In [ ]:
hep14 = p4_analysis.run_hep14(scores)
hep14

### Final primary-test interpretation

The HEP-14 primary-test protocol supports lower XGBoost fold-level log loss versus Random Forest, Decision Tree, KNN, and Naive Bayes after Bonferroni correction. For Logistic Regression, Shapiro-Wilk gives p ≈ 0.01423, so the primary test is two-sided exact Wilcoxon: raw p = 0.0625 and adjusted p = 0.3125. This is insufficient evidence that XGBoost is better under the primary protocol and does not establish equivalence.

The paired t-test/t-based CI and Wilcoxon can disagree because they use different assumptions and sensitivities. With n=5, Shapiro-Wilk has very low power, while two-sided exact Wilcoxon has coarse p-value resolution: even five same-direction differences have minimum p = 0.0625, which cannot reach the Bonferroni adjusted alpha 0.01. CV training sets also overlap, so fold-level p-values require cautious interpretation.

## 4. XGBoost diagnostic OOF attempt and reproduction gate

In [ ]:
oof = pd.read_csv(PROJECT_ROOT / 'results' / 'diagnostics' / 'oof_xgboost_reproduction_attempt.csv')
official_folds = pd.read_csv(PROJECT_ROOT / 'data' / 'interim' / 'fold_id.csv')
oof_validation = p4_analysis.validate_oof_xgboost(oof, official_folds)
gate = json.loads((PROJECT_ROOT / 'results' / 'validation' / 'xgboost_oof_reproduction_gate.json').read_text())
reproduction = pd.read_csv(PROJECT_ROOT / 'results' / 'validation' / 'xgboost_oof_reproduction.csv')
oof_validation, gate, reproduction

## 5. HEP-15 — calibration gate (diagnostic attempt is not approved OOF)

In [ ]:
if gate['status'] == 'PASS':
    label_map = {name: index for index, name in enumerate(stats.CLASS_ORDER)}
    y_true = oof['y_true'].map(label_map).to_numpy(dtype=int)
    y_proba = oof[['prob_C', 'prob_CL', 'prob_D']].to_numpy(dtype=float)
    calibration_metrics = {
        'multiclass_brier_score': stats.brier_multiclass(y_true, y_proba),
        'top_label_ece': stats.top_label_ece(y_true, y_proba, n_bins=10),
        'macro_classwise_ece': stats.macro_classwise_ece(y_true, y_proba, n_bins=10),
    }
    figure, _, top_bins, class_bins = stats.plot_reliability_diagram(y_true, y_proba, n_bins=10)
    print(calibration_metrics)
else:
    print('HEP-15 BLOCKED: reproduction gate failed; no final calibration result is reported.')

## 6. Limitations

- The comparison protocol was approved after results were available and is not preregistered.
- Hyperparameter selection and evaluation reused the same fixed folds rather than nested CV.
- Five folds provide very low power for distributional checks.
- Fold training sets overlap, weakening independence assumptions.
- Calibration is reported only if the frozen-pipeline OOF reproduction gate passes. No calibrator is fit and evaluated on the same OOF data.